# 01 · Data understanding

**Synthetic public sample · Milestone 1 observation only.** All substantive public tables are loaded at a pinned revision. Verify schema and coverage before interpreting absence. No ML or churn modeling.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/data/load.py").exists(), "Run from the repository or notebooks directory"
sys.path.insert(0, str(ROOT))
from src.data.load import load_tables, audit_tables
from src.features.attendance import member_metrics
tables = load_tables(ROOT)
m, c = tables["members"], tables["checkins"]


In [2]:
manifest = (ROOT / "data/raw/manifest.json").read_text()
print(manifest)
for name, frame in tables.items():
    print(name, frame.shape)
    display(pd.DataFrame({"type": frame.dtypes.astype(str), "nulls": frame.isna().sum(), "distinct_nonnull": frame.nunique()}))

{
  "dataset": "mindweave/gym-memberships",
  "revision": "3ff8e2fd4b429b5d0d4faf79344b3c23bc00fe5e",
  "synthetic": true,
  "license": "CC-BY-NC-4.0",
  "scope": "public free sample",
  "retrieved_at_utc": "2026-10-13T12:56:52.139426+00:00",
  "files": [
    {
      "source_path": "README.md",
      "local_path": "dataset_card.md",
      "url": "https://huggingface.co/datasets/mindweave/gym-memberships/resolve/3ff8e2fd4b429b5d0d4faf79344b3c23bc00fe5e/README.md",
      "bytes": 2429,
      "sha256": "556701a64b7002b6decf71ceff9e355bca243a9d9397536c5b10aa9e78130dd4"
    },
    {
      "source_path": "data/checkins.csv",
      "local_path": "checkins.csv",
      "url": "https://huggingface.co/datasets/mindweave/gym-memberships/resolve/3ff8e2fd4b429b5d0d4faf79344b3c23bc00fe5e/data/checkins.csv",
      "bytes": 64497,
      "sha256": "ec03d0578309f735aa7884e4da46610def075fc8473bbf089ac0ef1cb4b5cea3"
    },
    {
      "source_path": "data/members.csv",
      "local_path": "members.csv",
  

,type,nulls,distinct_nonnull
id,int64,0,300
first_name,str,0,15
last_name,str,0,15
email,str,0,300
phone,str,0,300
signup_date,datetime64[us],0,223
plan_id,int64,0,5
location_id,int64,0,3
is_active,bool,0,2
cancel_date,datetime64[us],160,125


checkins (2000, 5)


,type,nulls,distinct_nonnull
id,int64,0,2000
member_id,int64,0,32
location_id,int64,0,3
checkin_datetime,datetime64[us],0,1997
duration_minutes,int64,0,108


locations (3, 4)


,type,nulls,distinct_nonnull
id,int64,0,3
name,str,0,3
capacity,int64,0,3
has_pool,bool,0,2


plans (5, 4)


,type,nulls,distinct_nonnull
id,int64,0,5
name,str,0,5
price,float64,0,5
share,float64,0,5


## Key integrity and event plausibility

Hard audit failures block the pipeline. Same-day repeats and overlapping reported durations are retained as flags. Foreign-key joins are validated many-to-one.

In [3]:
audit = audit_tables(tables)
display(pd.Series(audit["checks"], name="failed_record_count").to_frame())
print(audit["observations"])
assert audit["all_hard_checks_pass"]

,failed_record_count
members_missing_pk,0
members_duplicate_pk,0
members_nonpositive_pk,0
checkins_missing_pk,0
checkins_duplicate_pk,0
checkins_nonpositive_pk,0
locations_missing_pk,0
locations_duplicate_pk,0
locations_nonpositive_pk,0
plans_missing_pk,0


{'checkins_on_cancel_day': 0, 'visits_away_from_home_branch': 0, 'multi_visit_member_days': 252, 'overlapping_sessions': 48, 'observed_member_ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]}


In [4]:
display(pd.DataFrame({"signup": [m.signup_date.min(), m.signup_date.max()], "checkin": [c.checkin_datetime.min(), c.checkin_datetime.max()], "cancel": [m.cancel_date.min(), m.cancel_date.max()]}, index=["min", "max"]))
print("Roster:", len(m), "Observed members:", c.member_id.nunique(), "Unrepresented:", len(m)-c.member_id.nunique())
print("Cancel dates:", m.cancel_date.notna().sum(), "Inactive snapshots:", (~m.is_active).sum())
display(tables["locations"])
display(tables["plans"])

,signup,checkin,cancel
min,2023-01-01,2023-01-12 18:26:00,2023-02-01
max,2024-12-21,2024-12-28 09:33:00,2024-12-28


Roster: 300 Observed members: 32 Unrepresented: 268
Cancel dates: 140 Inactive snapshots: 140


,id,name,capacity,has_pool
0,1,IronFit Downtown,150,True
1,2,IronFit Eastside,100,False
2,3,IronFit Pearl District,120,True


,id,name,price,share
0,1,Basic Monthly,29.99,0.35
1,2,Premium Monthly,49.99,0.25
2,3,Basic Annual,299.99,0.20
3,4,Premium Annual,499.99,0.12
4,5,Student,19.99,0.08


## Interpretation

Cancellation dates/reasons and is_active exist publicly, but they are contractual fields, not engagement labels. Event IDs 1–2000 cover member IDs 1–32; absence for 268 members is not inactivity. No public class-booking or operational anomaly event table exists. Closure and promotion are described without dates/branches or ground truth. See docs/data_dictionary.md, docs/entity_relationships.md and docs/dataset_limitations.md.